# 13. Brillouin Zones, Catalysis Volcanos, Defects & LOBSTER COHP

This tutorial demonstrates four domain workflows plus Interactive 3D Studio 2.0:

1. **3D First Brillouin Zones** (`compute_brillouin_zone`, `plot_brillouin_zone`)
2. **LOBSTER `-pCOHP` & `-ICOHP` Bonding Curves** (`read_cohpcar`, `plot_cohp`)
3. **Sabatier Catalysis Volcanos** (`plot_volcano`) & **Semiconductor Defect Formation** (`plot_defect_formation`)
4. **Interactive 3D Studio 2.0** trajectory playback and CUBE orbital isosurfaces

In [ ]:
import matplotlib.pyplot as plt

import xkcd_matchem as xm
from xkcd_matchem.render import interactive

## 1. 3D First Brillouin Zones & High-Symmetry k-Paths

Just as a crystal repeats in real space inside a unit cell, electron and phonon waves repeat in reciprocal (wavevector) space inside the **first Brillouin zone**—the Wigner-Seitz polyhedron enclosing all **k**-points closer to the zone center $\Gamma$ than to any other reciprocal lattice point. Tracing a **high-symmetry k-path** (like $\Gamma \to X \to W \to K \to \Gamma \to L$ in FCC silicon) visits the special corners and face centers where band edges usually occur. `compute_brillouin_zone` automatically detects primitive centering (F/I/C), reduces the reciprocal cell, constructs the polyhedron via `HalfspaceIntersection`, and maps standard k-paths.

In [ ]:
si = xm.load("si_diamond.cif")
bz_si = xm.compute_brillouin_zone(si)
print(f"Si BZ family: {bz_si.lattice_system}, facets: {len(bz_si.facets)}")

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.8))
    xm.plot_brillouin_zone(
        "si_diamond.cif",
        ax=axes[0],
        title="Silicon (FCC Truncated Octahedron)",
    )
    xm.plot_brillouin_zone(
        "mos2_2h.poscar",
        ax=axes[1],
        color="#009E73",
        title="2H-MoS2 (Hexagonal Prism BZ)",
    )
    fig.tight_layout()
    plt.show()

## 2. LOBSTER `-pCOHP` & `-ICOHP` Chemical Bonding

**Crystal Orbital Hamilton Population (COHP)** breaks a solid's electronic density of states down into pairwise chemical bonds. Following standard chemical convention (`minus_cohp=True`), positive `-pCOHP` lobes on the right mark stabilizing **bonding** states, negative lobes on the left mark destabilizing **antibonding** states, and the cumulative integral `-ICOHP` up to the Fermi level $E_F$ gauges overall covalent bond strength. Load LOBSTER `COHPCAR.lobster` / `COOPCAR.lobster` files directly via `xm.load(...)` or `xm.read_cohpcar(...)`.

In [ ]:
cohp = xm.load("si_cohp.lobster")
print("Interactions:", list(cohp.cohp.keys()))

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.0))
    xm.plot_cohp(
        cohp,
        ax=axes[0],
        title="Differential -pCOHP (Si-Si)",
    )
    xm.plot_cohp(
        cohp,
        integrated=True,
        ax=axes[1],
        title="Integrated -ICOHP (Si-Si)",
    )
    fig.tight_layout()
    plt.show()

## 3. Sabatier Catalysis Volcanos & Defect Formation Diagrams

- **Sabatier catalysis volcanos** (`plot_volcano`) plot catalytic activity against an adsorbate binding-energy descriptor ($\Delta G_{\mathrm{ads}}$): surfaces that bind intermediates too strongly get poisoned (strong-binding leg), while surfaces that bind too weakly fail to activate reactants (weak-binding leg)—peaking at the optimal catalyst near the apex.
- **Semiconductor defect formation diagrams** (`plot_defect_formation`) track how the energy cost of creating a point defect (like a silicon vacancy $V_{\mathrm{Si}}$) varies linearly with the Fermi level, $E^f(E_F) = E_0 + q\,E_F$, where the slope equals the defect charge $q$. Evaluating the lowest-energy envelope across all charge states produces **charge-transition kinks `(q1/q2)`** marking the Fermi levels where the defect switches from charge state $q_1$ to $q_2$.

In [ ]:
with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
    xm.plot_volcano(
        [-0.75, -0.35, -0.05, 0.25, 0.65],
        [-2.9, -1.35, -0.35, -1.25, -3.1],
        labels=["Ru", "Rh", "Pt", "Pd", "Au"],
        ax=axes[0],
        title="HER Sabatier Volcano",
    )
    xm.plot_defect_formation(
        {
            "V_Si": [(2, 2.0), (1, 2.35), (0, 3.05)],
            "Si_i": [(1, 2.5), (0, 2.95), (-1, 3.85)],
            "O_i": [(0, 2.4), (-1, 2.95), (-2, 3.85)],
        },
        band_gap=1.12,
        ax=axes[1],
        title="Si Defect Formation Energies",
    )
    fig.tight_layout()
    plt.show()

## 4. Interactive 3D Studio 2.0 (Trajectories & Orbital Cubes)

`xm.render.serialize_structure`, `xm.render.build_interactive_html`, and `xm.view_structure` package multi-frame `Trajectory` objects, 3D `VolumetricGrid` orbital isosurface lobes, and force/spin vector arrows into self-contained offline HTML—ready for Jupyter notebooks or standalone browser sharing.

In [ ]:
traj_payload = interactive.serialize_structure(
    xm.load_trajectory("sn2_neb.xyz")
)
cube_payload = interactive.serialize_structure("water_homo.cube")
print("SN2 NEB studio frames:", len(traj_payload.get("frames", [])))
print("Water HOMO isosurfaces:", len(cube_payload.get("isosurfaces", [])))

xm.view_structure("water_homo.cube", backend="matplotlib")
plt.show()